In [1]:
import os
import pandas as pd
from dateutil import parser
import re

def normalize_timestamp_for_merge(timestamp_str):
    """
    Attempt to parse timestamp.
    If successful, return formatted time string.
    If failed, return None for subsequent marking.
    """
    if pd.isna(timestamp_str) or str(timestamp_str).strip() == '':
        return None

    timestamp_str = str(timestamp_str).strip()

    # Reuse all previous preprocessing logic here
    cleaned_str = timestamp_str
    cleaned_str = re.sub(r'[·、，。；：""''（）【】]', ' ', cleaned_str)
    cleaned_str = re.sub(r'晚上(\d{1,2}:\d{2})', lambda m: str(int(m.group(1).split(':')[0]) + 12) + ':' + m.group(1).split(':', 1)[1], cleaned_str)
    cleaned_str = re.sub(r'下午(\d{1,2}:\d{2})', lambda m: str(int(m.group(1).split(':')[0]) + 12) + ':' + m.group(1).split(':', 1)[1], cleaned_str)
    cleaned_str = re.sub(r'上午(\d{1,2}:\d{2})', r'\1', cleaned_str)
    cleaned_str = re.sub(r'(\d{4})年(\d{1,2})月(\d{1,2})日', r'\1-\2-\3', cleaned_str)
    cleaned_str = re.sub(r'[;;]', ':', cleaned_str)
    cleaned_str = re.sub(r'(\d{4}-\d{1,2}-\d{1,2})(\d{1,2}:\d{2})', r'\1 \2', cleaned_str)
    cleaned_str = re.sub(r'[^\dA-Za-z:\s-]', ' ', cleaned_str)
    cleaned_str = re.sub(r'\s+', ' ', cleaned_str).strip()

    try:
        parsed_datetime = parser.parse(cleaned_str, fuzzy=True)
        return parsed_datetime.strftime('%Y-%m-%d %H:%M:%S')
    except (parser.ParserError, ValueError, OverflowError):
        # Parsing failed, return None
        return None

def process_excel_files_with_merge(directory="."):
    """
    Process all .xlsx files in the specified directory:
    1. Merge unparseable timeStamp content into the text column.
    2. Use forward fill to fill a time for these rows.
    """
    for filename in os.listdir(directory):
        if filename.endswith(".xlsx"):
            file_path = os.path.join(directory, filename)
            print(f"\nProcessing file: {file_path}...")

            try:
                df = pd.read_excel(file_path, engine='openpyxl')

                if 'timeStamp' not in df.columns:
                    print(f"File {filename} does not have 'timeStamp' column, skipped.")
                    continue

                # Ensure 'text' column exists, create if not
                if 'text' not in df.columns:
                    df['text'] = ''
                # Convert 'text' column to string type to avoid type errors during merging
                df['text'] = df['text'].astype(str).fillna('')

                # Create a copy to store original timeStamp values for merging
                original_timestamps = df['timeStamp'].copy()

                # Step 1: Preprocess timeStamp column, mark invalid items (return None)
                df['timeStamp'] = df['timeStamp'].apply(normalize_timestamp_for_merge)

                # Step 2: Find all rows where timeStamp is None
                invalid_mask = df['timeStamp'].isna()

                if invalid_mask.any():
                    print(f"  Found {invalid_mask.sum()} invalid timestamps, proceeding with content merging and time filling...")

                    # Merge invalid timeStamp content into text column
                    # Use .loc to avoid SettingWithCopyWarning
                    df.loc[invalid_mask, 'text'] = df.loc[invalid_mask, 'text'] + " [Invalid Timestamp: " + original_timestamps[invalid_mask].astype(str) + "]"

                    # Step 3: Use forward fill (ffill) to fill invalid timestamps
                    df['timeStamp'] = df['timeStamp'].fillna(method='ffill')

                    # If there are still NaN after forward fill (e.g., first few timestamps in the file are invalid)
                    # Then use backward fill (bfill) to supplement
                    df['timeStamp'] = df['timeStamp'].fillna(method='bfill')

                    # If there are still NaN after the above two steps (entire column is invalid), use default time
                    if df['timeStamp'].isna().any():
                        default_time = '1970-01-01 00:00:00'
                        df['timeStamp'] = df['timeStamp'].fillna(default_time)
                        print(f"  Note: Some rows have been filled with default time '{default_time}' due to lack of valid time reference.")

                # Save modified file
                df.to_excel(file_path, index=False, engine='openpyxl')
                print(f"File {filename} processed and saved successfully.")

            except Exception as e:
                print(f"Critical error occurred while processing file {filename}: {e}")

if __name__ == "__main__":
    print("===== Starting to process Excel files, merging invalid timestamps and intelligently filling =====")
    process_excel_files_with_merge()
    print("\n===== All Excel files processed! =====")

===== Starting to process Excel files, merging invalid timestamps and intelligently filling =====

Processing file: .\#AI.xlsx...
File #AI.xlsx processed and saved successfully.

Processing file: .\#American.xlsx...
File #American.xlsx processed and saved successfully.

Processing file: .\#art.xlsx...
File #art.xlsx processed and saved successfully.

Processing file: .\#BBNaija.xlsx...
File #BBNaija.xlsx processed and saved successfully.

Processing file: .\#BillsMafia.xlsx...
File #BillsMafia.xlsx processed and saved successfully.

Processing file: .\#Bitcoin.xlsx...
File #Bitcoin.xlsx processed and saved successfully.

Processing file: .\#Covid19.xlsx...
File #Covid19.xlsx processed and saved successfully.

Processing file: .\#Epstein.xlsx...
File #Epstein.xlsx processed and saved successfully.

Processing file: .\#FC25.xlsx...
File #FC25.xlsx processed and saved successfully.

Processing file: .\#game.xlsx...
File #game.xlsx processed and saved successfully.

Processing file: .\#Han